In [83]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import sklearn

from mhealth_activity.recording import Recording
from mhealth_activity.types import Activity
from mhealth_activity import Trace

# Data exploration #

1. Find the step count ground truths

In [11]:
train_dir = Path("data/train")

gt_files = []

for path in sorted(train_dir.glob("*.pkl")):
    rec = Recording(str(path))
    if rec.labels is None:
        continue
    sc = rec.labels.get("step_count", None)
    if sc is not None and sc != -1:
        gt_files.append(path.name)

# save
with open("stepcount_files.json", "w") as f:
    json.dump(gt_files, f)

print(f"Saved {len(gt_files)} GT files")

Saved 33 GT files


In [13]:
rows = []

for name in gt_files:
    rec = Recording(str(train_dir / name))
    sc = int(rec.labels["step_count"])

    activity_ids = rec.labels.get("activities", [])
    activity_names = [
        activity.name.lower()
        for activity in Activity
        if activity.value in activity_ids
    ]

    phone_steps_trace = rec.data.get("phone_steps", None)
    phone_steps_val = (
        int(phone_steps_trace.values[-1])
        if phone_steps_trace is not None and len(phone_steps_trace.values) > 0
        else None
    )

    rows.append({
        "file": name,
        "gt_steps": sc,
        "phone_steps": phone_steps_val,
        "diff": None if phone_steps_val is None else phone_steps_val - sc,
        "activities": activity_names,
    })

df = pd.DataFrame(rows)
df

,file,gt_steps,phone_steps,diff,activities
0,train_trace_004.pkl,1023,971.0,-52.0,[walking]
1,train_trace_015.pkl,734,NaN,NaN,[]
2,train_trace_017.pkl,0,NaN,NaN,[cycling]
3,train_trace_021.pkl,0,NaN,NaN,[]
4,train_trace_054.pkl,1000,NaN,NaN,[]
5,train_trace_074.pkl,698,NaN,NaN,[]
6,train_trace_086.pkl,0,NaN,NaN,[cycling]
7,train_trace_090.pkl,155,97.0,-58.0,"[walking, cycling]"
8,train_trace_093.pkl,172,340.0,168.0,"[walking, cycling]"
9,train_trace_110.pkl,192,NaN,NaN,[]


# Windowed & peak-based baseline #

Estimate steps from accelerometer magnitude using local peak detection.

Method (brief):

- Compute a_mag, remove mean, bandpass (≈0.7–3 Hz)
- Split into short windows (~10 s)
- Detect peaks per window (distance + prominence)
- Keep windows with plausible cadence (≈1.3–2.8 Hz)
- Sum peaks → step count

In [97]:
from scipy.signal import butter, filtfilt, find_peaks
import numpy as np

def estimate_steps_windowed(
    rec,
    window_s=10.0,
    low_hz=0.7,
    high_hz=3.0,
    peak_prominence=0.14,
    max_step_hz=3.0,
    min_peak_rate_hz=1.3,
    min_std_threshold=0.08,
    min_final_steps_to_keep=80,
):
    ax = rec.data["ax"].values.astype(float)
    ay = rec.data["ay"].values.astype(float)
    az = rec.data["az"].values.astype(float)
    fs = float(rec.data["ax"].samplerate)

    mag = np.sqrt(ax**2 + ay**2 + az**2)
    mag_centered = mag - np.mean(mag)

    nyq = fs / 2.0
    b, a = butter(3, [low_hz / nyq, high_hz / nyq], btype="band")
    filt = filtfilt(b, a, mag_centered)

    win_len = int(window_s * fs)
    min_distance = int(fs / max_step_hz)

    total_steps = 0
    kept_windows = []

    for start in range(0, len(filt), win_len):
        end = min(start + win_len, len(filt))
        segment = filt[start:end]

        if len(segment) < max(10, min_distance):
            continue

        # energy-based gating
        seg_std = float(np.std(segment))
        if seg_std < min_std_threshold:
            continue

        peaks, props = find_peaks(
            segment,
            distance=max(1, min_distance),
            prominence=peak_prominence,
        )

        duration = len(segment) / fs
        peak_rate_hz = len(peaks) / max(duration, 1e-9)

        if peak_rate_hz > min_peak_rate_hz:
            total_steps += len(peaks)
            kept_windows.append((start, end, len(peaks), peak_rate_hz, seg_std))

    # global cleanup for tiny false positives
    if total_steps < min_final_steps_to_keep:
        total_steps = 0

    return {
        "steps_hat": int(total_steps),
        "filtered_signal": filt,
        "fs": fs,
        "kept_windows": kept_windows,
    }

In [98]:
rows = []

for name in gt_files:
    rec = Recording(str(train_dir / name))
    gt = int(rec.labels["step_count"])
    out = estimate_steps_windowed(rec)

    pred = out["steps_hat"]
    ape = abs(pred - gt) / max(gt, 1)

    rows.append({
        "file": name,
        "gt_steps": gt,
        "pred_steps": pred,
        "abs_err": abs(pred - gt),
        "ape": ape,
    })

peak_df = pd.DataFrame(rows).sort_values("gt_steps")
print(peak_df)

# --- summary metrics ---
mae = peak_df["abs_err"].mean()                # average absolute deviation
mape = peak_df.loc[peak_df["gt_steps"] > 0, "ape"].mean()

print("\nSummary:")
print(f"Average absolute deviation (MAE): {mae:.2f}")
print(f"Average relative deviation (MAPE): {mape:.3f}")

                   file  gt_steps  pred_steps  abs_err       ape
2   train_trace_017.pkl         0           0        0  0.000000
3   train_trace_021.pkl         0           0        0  0.000000
6   train_trace_086.pkl         0           0        0  0.000000
17  train_trace_185.pkl         0           0        0  0.000000
13  train_trace_131.pkl         0           0        0  0.000000
30  train_trace_361.pkl       147         152        5  0.034014
16  train_trace_161.pkl       154         159        5  0.032468
7   train_trace_090.pkl       155         155        0  0.000000
10  train_trace_116.pkl       168         232       64  0.380952
18  train_trace_200.pkl       168         224       56  0.333333
8   train_trace_093.pkl       172          90       82  0.476744
9   train_trace_110.pkl       192         255       63  0.328125
20  train_trace_212.pkl       245         268       23  0.093878
29  train_trace_354.pkl       255         250        5  0.019608
32  train_trace_388.pkl  

Note: the above logic scores 0.17028 on Kaggle public leaderboard

# ML-based window filtering #

Builds window-level features

In [84]:
# --- ML-Based Window Filtering: feature extraction ---
from scipy.signal import butter, filtfilt, find_peaks, welch
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import GroupKFold
import numpy as np
import pandas as pd

def extract_window_features(
    rec,
    file_name,
    gt_steps,
    window_s=10.0,
    low_hz=0.7,
    high_hz=3.0,
    peak_prominence=0.14,
    max_step_hz=3.0,
):
    ax = rec.data["ax"].values.astype(float)
    ay = rec.data["ay"].values.astype(float)
    az = rec.data["az"].values.astype(float)
    fs = float(rec.data["ax"].samplerate)

    mag = np.sqrt(ax**2 + ay**2 + az**2)
    mag_centered = mag - np.mean(mag)

    nyq = fs / 2.0
    b, a = butter(3, [low_hz / nyq, high_hz / nyq], btype="band")
    filt = filtfilt(b, a, mag_centered)

    win_len = int(window_s * fs)
    min_distance = int(fs / max_step_hz)

    rows = []

    for win_idx, start in enumerate(range(0, len(filt), win_len)):
        end = min(start + win_len, len(filt))
        segment = filt[start:end]

        if len(segment) < max(10, min_distance):
            continue

        peaks, props = find_peaks(
            segment,
            distance=max(1, min_distance),
            prominence=peak_prominence,
        )

        duration = len(segment) / fs
        peak_rate_hz = len(peaks) / max(duration, 1e-9)

        mean_val = float(np.mean(segment))
        std_val = float(np.std(segment))
        rms_val = float(np.sqrt(np.mean(segment**2)))
        range_val = float(np.max(segment) - np.min(segment))
        q25 = float(np.quantile(segment, 0.25))
        q75 = float(np.quantile(segment, 0.75))

        prom = props.get("prominences", np.array([]))
        prom_mean = float(np.mean(prom)) if len(prom) > 0 else 0.0
        prom_std = float(np.std(prom)) if len(prom) > 0 else 0.0
        prom_max = float(np.max(prom)) if len(prom) > 0 else 0.0

        freqs, power = welch(segment, fs=fs, nperseg=min(256, len(segment)))
        valid = (freqs >= low_hz) & (freqs <= high_hz)
        if np.any(valid):
            valid_freqs = freqs[valid]
            valid_power = power[valid]
            dom_freq = float(valid_freqs[np.argmax(valid_power)])
            band_power = float(np.sum(valid_power))
            power_share = valid_power / max(np.sum(valid_power), 1e-12)
            spec_entropy = float(-(power_share * np.log(power_share + 1e-12)).sum())
        else:
            dom_freq = 0.0
            band_power = 0.0
            spec_entropy = 0.0

        rows.append({
            "file": file_name,
            "window_idx": win_idx,
            "start_s": start / fs,
            "end_s": end / fs,
            "gt_steps_total": gt_steps,
            "peak_count": int(len(peaks)),
            "peak_rate_hz": peak_rate_hz,
            "mean": mean_val,
            "std": std_val,
            "rms": rms_val,
            "range": range_val,
            "q25": q25,
            "q75": q75,
            "prom_mean": prom_mean,
            "prom_std": prom_std,
            "prom_max": prom_max,
            "dom_freq": dom_freq,
            "band_power": band_power,
            "spec_entropy": spec_entropy,
        })

    return rows


# Build weakly labeled window dataset
zero_step_threshold = 0
high_step_threshold = 700   # strong positive recordings

window_rows = []

for name in gt_files:
    rec = Recording(str(train_dir / name))
    gt = int(rec.labels["step_count"])

    rows = extract_window_features(rec, name, gt_steps=gt)

    # weak labels
    if gt == zero_step_threshold:
        label = 0
    elif gt >= high_step_threshold:
        label = 1
    else:
        label = None

    for r in rows:
        r["label"] = label
        window_rows.append(r)

window_df = pd.DataFrame(window_rows)

train_windows_df = window_df[window_df["label"].notna()].copy()
train_windows_df["label"] = train_windows_df["label"].astype(int)

print("All windows:", len(window_df))
print("Weakly labeled windows:", len(train_windows_df))
print(train_windows_df["label"].value_counts())
train_windows_df.head()

All windows: 1389
Weakly labeled windows: 709
label
1    539
0    170
Name: count, dtype: int64


,file,window_idx,start_s,end_s,gt_steps_total,peak_count,peak_rate_hz,mean,std,rms,range,q25,q75,prom_mean,prom_std,prom_max,dom_freq,band_power,spec_entropy,label
0,train_trace_004.pkl,0,0.000000,9.996269,1023,12,1.200448,0.001441,0.222157,0.222162,1.060415,-0.092449,0.112453,0.732614,0.248593,1.026999,1.563083,0.056144,0.801146,1
1,train_trace_004.pkl,1,9.996269,19.992537,1023,19,1.900709,-0.004767,0.318710,0.318746,1.022067,-0.318295,0.309789,0.896992,0.060971,1.003593,1.563083,0.126092,0.756095,1
2,train_trace_004.pkl,2,19.992537,29.988806,1023,20,2.000747,0.006513,0.319045,0.319112,1.039400,-0.306774,0.323016,0.864316,0.117853,0.996525,2.344625,0.124135,0.748361,1
3,train_trace_004.pkl,3,29.988806,39.985075,1023,19,1.900709,-0.001451,0.341410,0.341413,1.066712,-0.340737,0.340040,0.956791,0.051734,1.042375,2.344625,0.142364,0.739188,1
4,train_trace_004.pkl,4,39.985075,49.981343,1023,19,1.900709,-0.004856,0.356548,0.356581,1.134648,-0.356987,0.348002,1.002928,0.061799,1.102373,1.563083,0.156652,0.747299,1


Train classifier to reject non-step windows

In [85]:
# --- train window classifier ---
feature_cols = [
    "peak_count",
    "peak_rate_hz",
    "mean",
    "std",
    "rms",
    "range",
    "q25",
    "q75",
    "prom_mean",
    "prom_std",
    "prom_max",
    "dom_freq",
    "band_power",
    "spec_entropy",
]

X = train_windows_df[feature_cols].fillna(0.0)
y = train_windows_df["label"].values
groups = train_windows_df["file"].values

clf = ExtraTreesClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=3,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

cv = GroupKFold(n_splits=5)
fold_scores = []

for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y, groups), 1):
    clf.fit(X.iloc[tr_idx], y[tr_idx])
    score = clf.score(X.iloc[va_idx], y[va_idx])
    fold_scores.append(score)
    print(f"Fold {fold}: accuracy = {score:.3f}")

print(f"\nMean CV accuracy: {np.mean(fold_scores):.3f} ± {np.std(fold_scores):.3f}")

# fit final classifier on all weakly labeled windows
clf.fit(X, y)

feat_imp = pd.Series(clf.feature_importances_, index=feature_cols).sort_values(ascending=False)
feat_imp.head(10)

Fold 1: accuracy = 0.957
Fold 2: accuracy = 0.926
Fold 3: accuracy = 0.964
Fold 4: accuracy = 0.986
Fold 5: accuracy = 0.978

Mean CV accuracy: 0.962 ± 0.021


peak_rate_hz    0.195828
peak_count      0.180592
dom_freq        0.180560
q25             0.074524
band_power      0.055588
q75             0.054282
prom_mean       0.051837
std             0.049995
rms             0.043033
range           0.033723
dtype: float64

Recompute total steps with just accepted windows

In [86]:
# --- hybrid estimator: peak counting + ML window filter ---
def estimate_steps_hybrid(
    rec,
    clf,
    feature_cols,
    window_s=10.0,
    low_hz=0.7,
    high_hz=3.0,
    peak_prominence=0.14,
    max_step_hz=3.0,
    accept_proba_threshold=0.50,
    min_final_steps_to_keep=80,
):
    file_name = "current_recording"
    gt_dummy = -1

    rows = extract_window_features(
        rec=rec,
        file_name=file_name,
        gt_steps=gt_dummy,
        window_s=window_s,
        low_hz=low_hz,
        high_hz=high_hz,
        peak_prominence=peak_prominence,
        max_step_hz=max_step_hz,
    )

    if len(rows) == 0:
        return {
            "steps_hat": 0,
            "window_df": pd.DataFrame(),
        }

    wdf = pd.DataFrame(rows)
    Xw = wdf[feature_cols].fillna(0.0)

    proba = clf.predict_proba(Xw)[:, 1]
    pred_label = (proba >= accept_proba_threshold).astype(int)

    wdf["accept_proba"] = proba
    wdf["accept"] = pred_label

    total_steps = int(wdf.loc[wdf["accept"] == 1, "peak_count"].sum())

    # optional final cleanup for tiny false positives
    if total_steps < min_final_steps_to_keep:
        total_steps = 0

    return {
        "steps_hat": total_steps,
        "window_df": wdf,
    }

Compare to previous non-ML baseline

In [87]:
# --- compare baseline vs hybrid ---
comparison_rows = []

for name in gt_files:
    rec = Recording(str(train_dir / name))
    gt = int(rec.labels["step_count"])

    baseline_out = estimate_steps_windowed(rec)
    hybrid_out = estimate_steps_hybrid(rec, clf=clf, feature_cols=feature_cols)

    baseline_pred = int(baseline_out["steps_hat"])
    hybrid_pred = int(hybrid_out["steps_hat"])

    comparison_rows.append({
        "file": name,
        "gt_steps": gt,
        "baseline_pred": baseline_pred,
        "hybrid_pred": hybrid_pred,
        "baseline_abs_err": abs(baseline_pred - gt),
        "hybrid_abs_err": abs(hybrid_pred - gt),
        "baseline_ape": abs(baseline_pred - gt) / max(gt, 1),
        "hybrid_ape": abs(hybrid_pred - gt) / max(gt, 1),
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values("gt_steps")
print(comparison_df)

baseline_mae = comparison_df["baseline_abs_err"].mean()
hybrid_mae = comparison_df["hybrid_abs_err"].mean()

baseline_mape = comparison_df.loc[comparison_df["gt_steps"] > 0, "baseline_ape"].mean()
hybrid_mape = comparison_df.loc[comparison_df["gt_steps"] > 0, "hybrid_ape"].mean()

print("\nSummary:")
print(f"Baseline MAE : {baseline_mae:.2f}")
print(f"Hybrid MAE   : {hybrid_mae:.2f}")
print(f"Baseline MAPE: {baseline_mape:.3f}")
print(f"Hybrid MAPE  : {hybrid_mape:.3f}")

                   file  gt_steps  baseline_pred  hybrid_pred  \
2   train_trace_017.pkl         0             32            0   
3   train_trace_021.pkl         0              0            0   
6   train_trace_086.pkl         0             57            0   
17  train_trace_185.pkl         0              0            0   
13  train_trace_131.pkl         0              0            0   
30  train_trace_361.pkl       147            152           92   
16  train_trace_161.pkl       154            159          159   
7   train_trace_090.pkl       155            155            0   
10  train_trace_116.pkl       168            232          181   
18  train_trace_200.pkl       168            224          180   
8   train_trace_093.pkl       172            133            0   
9   train_trace_110.pkl       192            255          265   
20  train_trace_212.pkl       245            268          241   
29  train_trace_354.pkl       255            250          250   
32  train_trace_388.pkl  

submission

In [99]:
# --- step-count-only baseline submission ---
import re
from pathlib import Path
import pandas as pd

test_dir = Path("data/test")
submission_name = "submission_step_count_1.csv"

def parse_trace_id(path: Path) -> int:
    match = re.search(r"(\d+)\.pkl$", path.name)
    if match is None:
        raise ValueError(f"Could not parse Id from filename: {path.name}")
    return int(match.group(1))

rows = []

for path in sorted(test_dir.glob("*.pkl")):
    rec = Recording(str(path))
    step_pred = int(estimate_steps_windowed(rec)["steps_hat"])

    rows.append({
        "Id": parse_trace_id(path),
        "watch_loc": 0,      # default placeholder
        "path_idx": 0,       # default placeholder
        "standing": False,   # default placeholder
        "walking": False,    # default placeholder
        "running": False,    # default placeholder
        "cycling": False,    # default placeholder
        "step_count": step_pred,
    })

submission_df = pd.DataFrame(rows).sort_values("Id")

# optional sanity checks
print(submission_df.head())
print(f"\nRows: {len(submission_df)}")
print(f"Step count min/max: {submission_df['step_count'].min()} / {submission_df['step_count'].max()}")

submission_df.to_csv(submission_name, index=False)
print(f"\nSaved submission to {submission_name}")

   Id  watch_loc  path_idx  standing  walking  running  cycling  step_count
0   0          0         0     False    False    False    False         975
1   1          0         0     False    False    False    False         253
2   2          0         0     False    False    False    False         376
3   3          0         0     False    False    False    False        1093
4   4          0         0     False    False    False    False        1012

Rows: 280
Step count min/max: 0 / 1430

Saved submission to submission_step_count_1.csv
